# Deep kernel learning predicts complex traits and prioritizes candidate loci from pangenome structural variation in tomato

One fixed-partition experiment · 36 traits · 324 accessions. Each capacity ZIP contains 36 complete inference networks and their training records. Select **Runtime → Change runtime type → GPU** for training. Training and scoring are disabled by default.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/wyqmath/SVDKL.git"
REPO_DIR = Path("/content/SVDKL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
required = ["src/model.py", "src/train.py", "src/predict.py", "src/score.py", "src/prepare_data.py"]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError("The repository is incomplete: " + ", ".join(missing))
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import importlib.util
try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    in_colab = False
packages = ["numpy==2.3.5", "scipy==1.17.1", "torch==2.8.0", "threadpoolctl==3.6.0"]
if in_colab and importlib.util.find_spec("torch") is not None:
    packages = [line for line in packages if not line.lower().startswith("torch==")]
    print("Retaining Colab's installed PyTorch/CUDA build.")
packages.append("gdown")
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
import importlib.metadata
import numpy as np
import torch
from src.train import get_benchmark

plan = get_benchmark()
for package in ("numpy", "scipy", "torch", "threadpoolctl", "gdown"):
    print(package, importlib.metadata.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Model and inputs

Choose `SVDKL-0.01M_weights.zip`, `SVDKL-4.35M_weights.zip` or `SVDKL-46.46M_weights.zip`. Each includes 36 complete inference networks, 36 records, an index, the benchmark input snapshot and the marker catalogue. Open large archives with **Existing ZIP path**, or use **Upload ZIP**. Keep the extracted `weights` layout.

For input reconstruction, use [706.sv.vcf.gz](http://solomics.agis.org.cn/tomato/ftp/genotypes/706.sv.vcf.gz) and [pheno_exp_meta.txt](http://solomics.agis.org.cn/tomato/ftp/expression/pheno_exp_meta.txt) from [Zhou et al. (2022)](https://doi.org/10.1038/s41586-022-04808-9). Trait identities follow [Zhu et al. (2018)](https://doi.org/10.1016/j.cell.2017.12.019).


In [ ]:
MODEL = "SVDKL-0.01M"  # @param ["SVDKL-0.01M", "SVDKL-4.35M", "SVDKL-46.46M"]
print("Selected capacity:", MODEL)


In [ ]:
import filecmp
import re
import shutil
import stat
import tempfile
import urllib.parse
import urllib.request
import zipfile
from pathlib import PurePosixPath


def unpack_zip(archive_path, root):
    with zipfile.ZipFile(archive_path) as archive, tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        staging, entries, seen = Path(temporary), [], set()
        if root == "weights":
            prefix = f"weights/rebuilt_fold0/{MODEL}"
            expected = {"weights/marker_ids.npy", "weights/_input_snapshot.npz", prefix + "/index.json"}
            for trait in plan["traits"]:
                stem = f"{prefix}/fold_0/{MODEL}_f0_{trait}"
                expected.update({stem + ".full.pt", stem + ".json"})
            members = [member.filename for member in archive.infolist() if not member.is_dir()]
            if len(members) != 75 or len(set(members)) != 75 or set(members) != expected:
                raise ValueError(f"Select the single-experiment {MODEL} ZIP containing 36 complete networks and records.")
        for member in archive.infolist():
            name = member.orig_filename
            parts = name.rstrip("/").split("/")
            mode = member.external_attr >> 16
            if ("\\" in name or "\x00" in name or PurePosixPath(name).is_absolute()
                    or parts[0] != root or any(p in {"", ".", ".."} or ":" in p or p.rstrip(". ") != p for p in parts)
                    or stat.S_ISLNK(mode) or stat.S_IFMT(mode) not in {0, stat.S_IFREG, stat.S_IFDIR}):
                raise ValueError(f"Unsafe ZIP entry: {name!r}")
            key = "/".join(parts).casefold()
            if key in seen or (len(parts) == 1 and not member.is_dir()):
                raise ValueError(f"Invalid or duplicate ZIP entry: {name!r}")
            if root == "weights" and len(parts) > 1:
                common = len(parts) == 2 and parts[1] in {"marker_ids.npy", "_input_snapshot.npz"}
                rebuilt = (parts[1] == "rebuilt_fold0" and
                           ((len(parts) == 2 and member.is_dir()) or
                            (len(parts) >= 3 and parts[2] == MODEL)))
                if not (common or rebuilt):
                    raise ValueError(f"Select the {MODEL} weights ZIP for the chosen capacity.")
            seen.add(key)
            if member.is_dir():
                continue
            source, target = staging.joinpath(*parts), REPO_DIR.joinpath(*parts)
            source.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as handle, source.open("xb") as output:
                shutil.copyfileobj(handle, output)
            if target.is_symlink() or any(parent.is_symlink() for parent in target.parents):
                raise ValueError(f"Refusing a symbolic-link destination: {name!r}")
            if target.exists() and (not target.is_file() or not filecmp.cmp(source, target, shallow=False)):
                raise FileExistsError(f"Conflicting existing file: {name!r}; no files were replaced.")
            entries.append((source, target))
        if not entries:
            raise ValueError("The ZIP contains no files.")
        for source, target in entries:
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                # Staging is on the same filesystem: publish without a second
                # full-size copy and never replace an existing target.
                os.link(source, target)
                source.unlink()


def download_to(url, target):
    parsed = urllib.parse.urlsplit(url.strip())
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Enter a public file URL or use file upload.")
    if parsed.hostname in {"drive.google.com", "docs.google.com"}:
        import gdown
        match = re.search(r"/file/d/([A-Za-z0-9_-]+)", parsed.path)
        file_id = match.group(1) if match else urllib.parse.parse_qs(parsed.query).get("id", [""])[0]
        if not re.fullmatch(r"[A-Za-z0-9_-]+", file_id):
            raise ValueError("Use a Google Drive file link, not a folder link.")
        if gdown.download(id=file_id, output=str(target), use_cookies=False) is None:
            raise RuntimeError("The Google Drive file could not be downloaded.")
    else:
        with urllib.request.urlopen(url.strip(), timeout=60) as response, target.open("xb") as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)


def receive_zip(label):
    from google.colab import files

    with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        archive_path = Path(temporary) / "archive.zip"
        print(f"Choose {label}")
        previous = Path.cwd()
        try:
            os.chdir(temporary)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1:
            raise ValueError("Upload one weights ZIP file.")
        archive_path.write_bytes(next(iter(uploaded.values())))
        unpack_zip(archive_path, "weights")


In [ ]:
WEIGHTS_SOURCE = "Skip"  # @param ["Skip", "Existing ZIP path", "Upload ZIP"]
EXISTING_ZIP_PATH = ""  # @param {type:"string"}

if WEIGHTS_SOURCE == "Existing ZIP path":
    if not EXISTING_ZIP_PATH.strip():
        raise ValueError("Enter the existing archive path, for example on a mounted Drive.")
    archive_path = Path(EXISTING_ZIP_PATH).expanduser()
    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)
    unpack_zip(archive_path, "weights")
    print("Loaded:", archive_path)
elif WEIGHTS_SOURCE == "Upload ZIP":
    receive_zip(f"{MODEL}_weights.zip")
else:
    print("Pretrained states skipped; original-file preparation and training remain available.")


## Input arrays

The default uses the embedded `weights/_input_snapshot.npz`. An existing prepared input or the original VCF and phenotype files can also be used. Reconstruction retains 324 accessions × 106,362 DELs and 36 traits, and compares all eight arrays with the snapshot when available.


In [ ]:
INPUT_MODE = "Saved-state input"  # @param ["Saved-state input", "Original files in data/", "Upload original files", "Download original files"]
VCF_URL = "http://solomics.agis.org.cn/tomato/ftp/genotypes/706.sv.vcf.gz"  # @param {type:"string"}
PHENOTYPES_URL = "http://solomics.agis.org.cn/tomato/ftp/expression/pheno_exp_meta.txt"  # @param {type:"string"}

snapshot = Path("weights/_input_snapshot.npz")
output = Path("data/dataset_inputs.npz")
prepared = None
if INPUT_MODE == "Saved-state input":
    if snapshot.is_file():
        prepared = snapshot
        print("Using the fixed original input supplied with the saved prediction states.")
    elif output.is_file():
        prepared = output
        print("Using the existing prepared input.")
    else:
        print("Load a weights ZIP above or select an original-file input option.")
else:
    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)
    names = {"706.sv.vcf.gz", "pheno_exp_meta.txt"}
    if INPUT_MODE == "Upload original files":
        from google.colab import files
        print("Choose 706.sv.vcf.gz and pheno_exp_meta.txt.")
        previous = Path.cwd()
        with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
            try:
                os.chdir(temporary)
                uploaded = files.upload()
            finally:
                os.chdir(previous)
            if not uploaded or not set(uploaded) <= names:
                raise ValueError("Upload only 706.sv.vcf.gz and pheno_exp_meta.txt.")
            for name, payload in uploaded.items():
                target = data_dir / name
                if target.exists():
                    raise FileExistsError(f"{target} already exists; use Original files in data/.")
                with target.open("xb") as handle:
                    handle.write(payload)
    elif INPUT_MODE == "Download original files":
        for name, url in (("706.sv.vcf.gz", VCF_URL), ("pheno_exp_meta.txt", PHENOTYPES_URL)):
            target = data_dir / name
            if not target.exists():
                with tempfile.TemporaryDirectory(dir=data_dir) as temporary:
                    downloaded = Path(temporary) / name
                    download_to(url, downloaded)
                    if target.exists():
                        raise FileExistsError(f"{target} appeared during download.")
                    downloaded.rename(target)
    missing = [name for name in sorted(names) if not (data_dir / name).is_file()]
    if missing:
        raise FileNotFoundError("Place these original files in data/: " + ", ".join(missing))
    if not output.exists():
        subprocess.run([sys.executable, "-m", "src.prepare_data",
                        "--vcf", "data/706.sv.vcf.gz", "--phenotypes", "data/pheno_exp_meta.txt",
                        "--output", str(output)], check=True)
    prepared = output

if prepared is not None:
    expected_keys = {"dosage", "accessions", "traits", "phenotypes", "marker_id", "chrom", "pos", "svlen"}
    with np.load(prepared, allow_pickle=False) as current:
        assert set(current.files) == expected_keys
        np.testing.assert_array_equal(current["accessions"], plan["accessions"])
        np.testing.assert_array_equal(current["traits"], plan["traits"])
        assert current["dosage"].shape == (324, 106362)
        assert current["phenotypes"].shape == (324, 36)
        if snapshot.is_file() and prepared.resolve() != snapshot.resolve():
            with np.load(snapshot, allow_pickle=False) as original:
                assert set(original.files) == expected_keys
                for key in expected_keys:
                    np.testing.assert_array_equal(current[key], original[key], err_msg=key)
            print("All eight reconstructed arrays match the independent original snapshot.")
    print("Benchmark input ready:", prepared)


## Train

Training is disabled by default. `TRAIN_TRAIT="all"` fits the 36 traits of the selected capacity on the fixed partition; enter one trait ID for a single case. New outputs include complete `.full.pt` checkpoints with AdamW state, prediction states and `.json` records. Resume verifies and skips completed cases; it does not continue an interrupted epoch. Distributed inference exports omit optimizer state and remain separate from these training outputs.


In [ ]:
RUN_TRAINING = False  # @param {type:"boolean"}
TRAIN_TRAIT = "all"  # @param {type:"string"}

if RUN_TRAINING:
    if prepared is None:
        raise RuntimeError("Choose or prepare the input arrays before training.")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    subprocess.run([sys.executable, "-m", "src.train", "--model", MODEL,
                    "--trait", TRAIN_TRAIT, "--fold", "0", "--device", device,
                    "--data", str(prepared), "--output", "results", "--save-optimizer", "--resume"], check=True)
else:
    print("Training disabled. Enable RUN_TRAINING to fit the selected traits.")


## Predict

Automatic selection checks new training outputs, then the pretrained complete network. The inference export retains the trained neural network, GP state and fitted preprocessing; shared inputs and marker identities remain in `weights/`. Prediction exports every supplied accession. Output filenames identify the model, trait and source.


In [ ]:
PREDICTION_SOURCE = "Automatic"  # @param ["Automatic", "Training output", "Pretrained weights"]
PREDICT_TRAIT = "SlFM0030"  # @param {type:"string"}


def find_state(root, trait):
    if trait not in plan["traits"]:
        raise ValueError("Trait is not in the fixed 36-trait experiment.")
    root = Path(root)
    folder = root / "fold_0" if root.name == MODEL else root / MODEL / "fold_0"
    path = folder / f"{MODEL}_f0_{trait}.full.pt"
    return path if path.is_file() else None


if PREDICTION_SOURCE == "Automatic":
    sources = ["results", "weights/rebuilt_fold0"]
elif PREDICTION_SOURCE == "Training output":
    sources = ["results"]
else:
    sources = ["weights/rebuilt_fold0"]
checkpoint = None
for source in sources:
    checkpoint = find_state(source, PREDICT_TRAIT)
    if checkpoint is not None:
        break
if prepared is None:
    print("Prediction skipped: choose or prepare the input arrays above.")
elif checkpoint is None:
    print(f"Prediction skipped: train {PREDICT_TRAIT} or load {MODEL}_weights.zip.")
else:
    fit_label = "pretrained" if source == "weights/rebuilt_fold0" else "training_output"
    basename = f"{fit_label}_{MODEL}_{PREDICT_TRAIT}"
    output = Path("predictions") / (basename + ".csv")
    run_number = 2
    while output.exists():
        output = Path("predictions") / f"{basename}_run{run_number}.csv"
        run_number += 1
    subprocess.run([sys.executable, "-m", "src.predict", "--checkpoint", str(checkpoint),
                    "--data", str(prepared), "--device", "cpu", "--output", str(output)], check=True)
    print("Prediction file:", output)
    import csv
    with output.open(newline="") as handle:
        for row in list(csv.reader(handle))[:6]:
            print(row)


## Evaluate

Evaluation uses the 36 complete networks and their records for the selected capacity. It scores each trait on its fixed held-out accessions and reports per-trait metrics and their equal-weight mean across traits.


In [ ]:
RUN_SCORING = False  # @param {type:"boolean"}
SCORE_SOURCE = "Pretrained weights"  # @param ["Pretrained weights", "Training output"]

if RUN_SCORING:
    if prepared is None:
        raise RuntimeError("Choose or prepare the input arrays before scoring.")
    result_root = Path("results") if SCORE_SOURCE == "Training output" else Path("weights/rebuilt_fold0")
    missing = []
    for trait in plan["traits"]:
        state = find_state(result_root, trait)
        record = state.with_name(state.name.removesuffix(".full.pt") + ".json") if state else None
        if state is None or not record.is_file():
            missing.append(trait)
    if missing:
        print(f"Scoring skipped: {len(missing)} of 36 cases are missing. Complete training or load {MODEL}_weights.zip.")
    else:
        score_label = "training_output" if SCORE_SOURCE == "Training output" else "pretrained"
        output = Path(f"metrics_{score_label}_{MODEL}.json")
        if output.exists():
            print("Existing metrics:", output)
        else:
            subprocess.run([sys.executable, "-m", "src.score", "--results", str(result_root),
                            "--model", MODEL, "--data", str(prepared), "--output", str(output)], check=True)
else:
    print("Scoring disabled. Enable RUN_SCORING when the 36 cases are available.")


Download newly generated outputs before disconnecting. Each capacity ZIP contains the 36 complete inference networks for this fixed-partition experiment. Original complete training checkpoints remain separately archived.
